# 14 — CIDER Score Stability Across Sequencing Depth

Validates that depth normalization (`cdr3_unique / sampleMedianDepth`) removes
the confounding effect of sequencing depth on the CIDER T-cell score.

**Panel A** — Raw CDR3 unique count vs sequencing depth (shows raw correlation)  
**Panel B** — CIDER score (depth-adjusted) vs sequencing depth (should show no correlation)  
**Panel C** — CIDER score by depth quartile (stability check across depth range)

Includes all T-cell group patients (ICI and non-ICI), pan-cancer, from `ready.rds`.
(Supersedes the former `16_sequencing_depth.ipynb`, which ran the same check on the
same pan-cancer population and is now redundant with this notebook.)

In [ ]:
library(ggplot2)
library(dplyr)
library(patchwork)

In [ ]:
args <- commandArgs(trailingOnly = FALSE)
script_path <- sub("--file=", "", args[grep("--file=", args)])
SCRIPT_DIR <- if (length(script_path) > 0) {
  normalizePath(dirname(script_path), mustWork = FALSE)
} else {
  getwd()
}

source(file.path(SCRIPT_DIR, "../helpers/map.r"))
source(file.path(SCRIPT_DIR, "../helpers/shortcuts.r"))
source(file.path(SCRIPT_DIR, "../helpers/config.r"))
source(file.path(SCRIPT_DIR, "../helpers/themes.r"))

## 1 — Data preparation

In [ ]:
# All T-cell group patients, pan-cancer (ICI and non-ICI).
# ready.rds duplicates every patient once with cohort_broad = 'ICI-Treated';
# deduplicate by sampleId so each patient is counted once.
# sampleMedianDepth from COBALT is used in score computation.
df <- readRDS(paste0(CIDER_PROJ_DIR, "ready.rds")) |>
  dplyr::filter(gp == "T-cell") |>
  dplyr::distinct(sampleId, .keep_all = TRUE) |>
  dplyr::select(sampleId, cdr3_unique, sampleMedianDepth,
                cider_score_depth_adj) |>  # cohort covariate not used downstream in this notebook
  dplyr::filter(!is.na(sampleMedianDepth), !is.na(cdr3_unique),
                sampleMedianDepth > 0) |>
  dplyr::mutate(
    cider_log2  = log2(cider_score_depth_adj + 1),
    cdr3_log   = log1p(cdr3_unique),
    depth_log  = log10(sampleMedianDepth),
    depth_q    = cut(sampleMedianDepth,
                     breaks   = quantile(sampleMedianDepth, probs = c(0, .25, .5, .75, 1)),
                     labels   = c("Q1", "Q2", "Q3", "Q4"),
                     include.lowest = TRUE)
  )

message("N = ", nrow(df), " samples")
message("Depth range: ", round(min(df$sampleMedianDepth), 0),
        "x — ", round(max(df$sampleMedianDepth), 0), "x")

## 2 — Pearson correlations

In [ ]:
# Pearson r between depth and raw/adjusted scores, on the log-transformed
# scale already used for df$depth_log / cdr3_log / cider_log2 above -- kept
# consistent with Pearson usage elsewhere in the paper (e.g. the gene-set
# and CIDER-vs-TMB correlation panels), rather than Spearman's rank-based
# rho. The log transform is what makes Pearson appropriate here: it
# linearizes the depth/score relationship so Pearson's linearity assumption
# holds, rather than correlating the raw, heavily right-skewed values.
r_raw  <- cor.test(df$depth_log, df$cdr3_log,  method = "pearson")
r_adj  <- cor.test(df$depth_log, df$cider_log2, method = "pearson")

fmt_r <- function(ct) {
  paste0("r = ", round(ct$estimate, 3),
         ", ", fmt_pval(ct$p.value))
}

message("Raw CDR3 vs depth:      ", fmt_r(r_raw))
message("Adj CIDER vs depth:     ", fmt_r(r_adj))

## 3 — Figures

In [ ]:
scatter_theme <- go_theme +
  ggplot2::theme(plot.subtitle = ggplot2::element_text(size = 8, colour = "grey40"))

# Shared axis scales.
# Display is clipped to 170x (via coord_cartesian below, which zooms without
# dropping data from the fitted lm/smooth); 160x is the last labelled break.
depth_x_scale <- ggplot2::scale_x_continuous(
  breaks = c(80, 100, 120, 140, 160),
  labels = paste0(c(80, 100, 120, 140, 160), "×"),
  name   = "COBALT Median Sequencing Depth"
)

depth_x_limit <- ggplot2::coord_cartesian(xlim = c(NA, 170))

score_y_scale <- ggplot2::scale_y_continuous(
  trans   = log10p1_trans,
  breaks  = c(0, 1, 10, 100),
  labels  = scales::label_number(),
  expand  = ggplot2::expansion(mult = c(0.05, 0.05))
)

# ── Panel A: raw CDR3 vs depth ────────────────────────────────────────────────
p_raw <- df |>
  ggplot2::ggplot(ggplot2::aes(x = sampleMedianDepth, y = cdr3_unique)) +
  ggplot2::geom_point(alpha = 0.3, size = 0.6, colour = "grey40") +
  ggplot2::geom_smooth(method = "lm", se = TRUE,
                       colour = "black", linewidth = 0.7) +
  depth_x_scale +
  score_y_scale +
  depth_x_limit +
  scatter_theme +
  ggplot2::labs(
    tag      = "A",
    title    = "CIDER T-cell Score (Unadjusted) vs Depth",
    subtitle = fmt_r(r_raw),
    y        = "CDR3 Unique Count"
  )

# ── Panel B: depth-adjusted CIDER vs depth ────────────────────────────────────
p_adj <- df |>
  ggplot2::ggplot(ggplot2::aes(x = sampleMedianDepth, y = cider_score_depth_adj)) +
  ggplot2::geom_point(alpha = 0.3, size = 0.6, colour = "grey40") +
  ggplot2::geom_smooth(method = "lm", se = TRUE,
                       colour = "black", linewidth = 0.7) +
  depth_x_scale +
  score_y_scale +
  depth_x_limit +
  scatter_theme +
  ggplot2::labs(
    tag      = "B",
    title    = "CIDER Score (Depth-Adjusted) vs Depth",
    subtitle = fmt_r(r_adj),
    y        = "CIDER T-cell Score"
  )

# ── Assemble ──────────────────────────────────────────────────────────────────
n_over_170 <- sum(df$sampleMedianDepth > 170, na.rm = TRUE)
message("Samples with depth > 170x (outside plotted range): ", n_over_170)

supplement_depth <-
  (p_raw | p_adj) +
  patchwork::plot_annotation(
    title   = "CIDER Score Stability Across Sequencing Depth",
    caption = paste0("Note: ", n_over_170, " sample(s) with depth >170× fall outside the plotted x-axis range (fitted lines use all samples)."),
    theme   = go_theme + ggplot2::theme(
      plot.tag     = ggplot2::element_text(size = 14, face = "bold"),
      plot.caption = ggplot2::element_text(size = 7, colour = "grey40", hjust = 1)
    )
  )

set_plot_dims(7.5, 3.5); supplement_depth

saveRDS(supplement_depth, paste0(CIDER_PROJ_DIR, "figs/supplement_depth_stability.rds"))
ggsave(paste0(CIDER_PROJ_DIR, "figs/supplement_depth_stability.png"),
       supplement_depth, width = 7.5, height = 3.5, dpi = 300)
message("Saved supplement_depth_stability.png")